In [1]:
import os
import re
import pickle
import pandas as pd

# 1. Lógica para encontrar o modelo MAIS RECENTE automaticamente
pasta_modelos = 'models'
arquivos_existentes = os.listdir(pasta_modelos)
versoes =[]

for arquivo in arquivos_existentes:
    match = re.search(r'motor_churn_v(\d+)\.pkl', arquivo)
    if match:
        versoes.append(int(match.group(1)))

if not versoes:
    print("Nenhum modelo encontrado na pasta 'models'. Rode o pipeline de treino primeiro!")
else:
    # Descobre o maior número (a versão mais nova)
    versao_recente = max(versoes)
    caminho_modelo = f'{pasta_modelos}/motor_churn_v{versao_recente}.pkl'
    
    print(f"🔄 Conectando ao servidor... Carregando o modelo: {caminho_modelo}")
    
    # 2. Descongelando (carregando) o cérebro do modelo
    with open(caminho_modelo, 'rb') as file:
        modelo_producao = pickle.load(file)
        
    print("✅ Motor carregado com sucesso e pronto para uso!\n")
    
    # ---------------------------------------------------------
    # 3. SIMULANDO O MUNDO REAL
    # Vamos puxar 3 linhas aleatórias da sua base para fingir 
    # que são 3 clientes navegando no site neste exato momento.
    # ---------------------------------------------------------
    print("📥 Recebendo dados de 3 clientes em tempo real...")
    df_novos_clientes = pd.read_csv('data/abt_churn.csv').sample(3, random_state=99)
    
    # O modelo não pode ver as colunas de ID e Data, então tiramos:
    X_novos = df_novos_clientes.drop(columns=['dtRef', 'idUsuario', 'flagChurn'])
    
    # 4. A Mágica Acontece: O modelo prevê tudo em milissegundos
    previsoes = modelo_producao.predict(X_novos)
    probabilidades = modelo_producao.predict_proba(X_novos)[:, 1] # Pega a propabilidade de ser 1 (Churn)
    
    # 5. Exibindo o alerta para a área de Negócios/Retenção
    print("\n" + "="*50)
    print("📊 PAINEL DE ALERTA DE CHURN (MUNDO REAL)")
    print("="*50)
    
    for i in range(len(previsoes)):
        id_cliente = df_novos_clientes.iloc[i]['idUsuario']
        
        # Cria uma mensagem visual baseada na previsão
        if previsoes[i] == 1:
            status = "🔴 ALTO RISCO DE EVASÃO (Ligar com oferta!)"
        else:
            status = "🟢 CLIENTE SEGURO (Não fazer nada)"
            
        print(f"👤 Cliente ID: {id_cliente}")
        print(f"   Decisão: {status}")
        print(f"   Probabilidade Matemática: {probabilidades[i]*100:.2f}%")
        print("-" * 50)

🔄 Conectando ao servidor... Carregando o modelo: models/motor_churn_v1.pkl
✅ Motor carregado com sucesso e pronto para uso!

📥 Recebendo dados de 3 clientes em tempo real...

📊 PAINEL DE ALERTA DE CHURN (MUNDO REAL)
👤 Cliente ID: b90bf9c7-24dc-469f-baaf-9ae992eee41d
   Decisão: 🟢 CLIENTE SEGURO (Não fazer nada)
   Probabilidade Matemática: 40.24%
--------------------------------------------------
👤 Cliente ID: 1614574e-7b0c-4a7c-90d9-ac53b7cf885f
   Decisão: 🔴 ALTO RISCO DE EVASÃO (Ligar com oferta!)
   Probabilidade Matemática: 74.36%
--------------------------------------------------
👤 Cliente ID: c6e08c3e-e13f-4cb4-ae2e-55f6ad790446
   Decisão: 🟢 CLIENTE SEGURO (Não fazer nada)
   Probabilidade Matemática: 48.72%
--------------------------------------------------
